# Crowd Anomaly Detection Training Notebook

This notebook follows an 18-section pipeline to train a crowd anomaly classifier with a fully manual LSTM implementation (no `nn.LSTM`).

In [ ]:
# SECTION 1 - Runtime Setup, Dependencies, and Reproducibility
# SECTION 2 - Global CONFIG Dictionary and Path Initialization
# SECTION 3 - Feature Schema and Utility Functions (118-D Per Frame)

# Install runtime dependencies in Colab.
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install opencv-python-headless mediapipe scikit-learn numpy pandas matplotlib tqdm Pillow

import os
import re
import json
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import matplotlib.pyplot as plt
import cv2
import mediapipe as mp

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from tqdm.auto import tqdm

# Reproducibility settings.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Central config dictionary required by the prompt.
CONFIG = {
    "epochs": 50,
    "lr": 0.001,
    "weight_decay": 1e-4,
    "patience": 8,
    "min_delta": 0.001,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "save_path": "/content/models/",
    "model_name": "crowd_anomaly_lstm_v1.pt",
    "sequence_length": 30,
    "batch_size": 32,
    "stride": 15,
    "rwf_fps": 5,
}

PATHS = {
    "godseye": Path("/content/godseye"),
    "rwf2000": Path("/content/rwf2000"),
    "ucf_crime": Path("/content/ucf_crime"),
    "models": Path(CONFIG["save_path"]),
}
PATHS["models"].mkdir(parents=True, exist_ok=True)

print("Resolved paths:")
for k, v in PATHS.items():
    print(f"  {k}: {v}")
print(f"Device: {CONFIG['device']}")

# Build feature schema: 99 + 17 + 1 + 1 = 118.
FEATURE_NAMES = []
for i in range(33):
    FEATURE_NAMES.extend([f"kp{i}_x", f"kp{i}_y", f"kp{i}_z"])
FEATURE_NAMES.extend([f"joint_angle_{i+1}" for i in range(17)])
FEATURE_NAMES.extend(["optical_flow_mag", "crowd_density"])
assert len(FEATURE_NAMES) == 118

# 17 angle triplets (indexing MediaPipe 33 landmark convention).
ANGLE_TRIPLETS = [
    (11, 13, 15), (12, 14, 16), (23, 25, 27), (24, 26, 28),
    (11, 23, 25), (12, 24, 26), (13, 11, 23), (14, 12, 24),
    (15, 13, 11), (16, 14, 12), (27, 25, 23), (28, 26, 24),
    (23, 11, 13), (24, 12, 14), (25, 23, 11), (26, 24, 12),
    (11, 12, 24),
]

def _angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    ba = a - b
    bc = c - b
    nba = np.linalg.norm(ba)
    nbc = np.linalg.norm(bc)
    if nba < 1e-8 or nbc < 1e-8:
        return 0.0
    cosv = float(np.dot(ba, bc) / (nba * nbc))
    cosv = max(-1.0, min(1.0, cosv))
    return float(np.degrees(np.arccos(cosv)))

def normalize_kps(kps) -> np.ndarray:
    arr = np.asarray(kps, dtype=np.float32).reshape(-1)
    out = np.zeros((99,), dtype=np.float32)
    out[: min(99, arr.size)] = arr[:99]
    return out.reshape(33, 3)

def zero_feature() -> np.ndarray:
    return np.zeros((118,), dtype=np.float32)

def build_frame_feature(curr_kps, prev_kps=None, density=1.0) -> np.ndarray:
    curr = normalize_kps(curr_kps)
    prev = np.zeros_like(curr) if prev_kps is None else normalize_kps(prev_kps)
    coords = curr.flatten().tolist()
    angles = [_angle(curr[i], curr[j], curr[k]) for (i, j, k) in ANGLE_TRIPLETS]
    flow = float(np.mean(np.abs(curr - prev)))
    vec = coords + angles + [flow, float(density)]
    if len(vec) != 118:
        raise ValueError(f"Feature size mismatch: {len(vec)}")
    return np.asarray(vec, dtype=np.float32)

def to_fixed_sequence(features, seq_len=30) -> np.ndarray:
    if len(features) >= seq_len:
        return np.stack(features[:seq_len], axis=0).astype(np.float32)
    pad = [zero_feature() for _ in range(seq_len - len(features))]
    return np.stack(pad + features, axis=0).astype(np.float32)

def sliding_windows(features, seq_len=30, stride=15):
    if len(features) == 0:
        return []
    if len(features) < seq_len:
        return [to_fixed_sequence(features, seq_len)]
    out = []
    for s in range(0, len(features) - seq_len + 1, stride):
        out.append(np.stack(features[s:s+seq_len], axis=0).astype(np.float32))
    return out

In [ ]:
# SECTION 4 - GodsEye JSON Parser and Sequence Builder (Source 1)
# SECTION 5 - RWF-2000 Video Loader + MediaPipe Pose Pipeline (Source 2)
# SECTION 6 - UCF-Crime Frame Grouper + MediaPipe Pose Pipeline (Source 3)

def parse_godseye_json(json_path: Path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Flexible schema handling for several possible JSON layouts.
    if isinstance(data, list):
        frames = data
    elif isinstance(data, dict):
        frames = data.get("frames", data.get("data", data.get("annotations", [data])))
    else:
        frames = []

    parsed = []
    for fr in frames:
        density = 1.0
        kp = None
        if isinstance(fr, dict):
            for key in ["person_count", "num_persons", "crowd_density"]:
                if key in fr:
                    try:
                        density = float(fr[key])
                    except Exception:
                        density = 1.0
            for key in ["keypoints", "pose_keypoints", "landmarks", "joints"]:
                if key in fr:
                    kp = fr[key]
                    break
            if kp is None:
                for key in ["persons", "people", "detections"]:
                    if key in fr and isinstance(fr[key], list) and len(fr[key]) > 0 and isinstance(fr[key][0], dict):
                        p0 = fr[key][0]
                        for k2 in ["keypoints", "pose_keypoints", "landmarks", "joints"]:
                            if k2 in p0:
                                kp = p0[k2]
                                density = float(max(len(fr[key]), 1))
                                break
                        break
        elif isinstance(fr, (list, tuple, np.ndarray)):
            kp = fr

        if kp is None:
            kp = np.zeros((33, 3), dtype=np.float32)
        parsed.append((normalize_kps(kp), density))

    return parsed


def load_source_godseye(root: Path):
    out = []
    counts = {"Fight": 0, "NonFight": 0}
    mapping = {"Fight": 1, "NonFight": 0}

    for cls, label in mapping.items():
        cls_dir = root / "train" / cls
        if not cls_dir.exists():
            continue

        for jf in tqdm(sorted(cls_dir.rglob("*.json")), desc=f"GodsEye {cls}", leave=False):
            try:
                rows = parse_godseye_json(jf)
                if len(rows) == 0:
                    continue
                feats = []
                prev = None
                for kps, density in rows:
                    feats.append(build_frame_feature(kps, prev, density))
                    prev = kps
                seq = to_fixed_sequence(feats, seq_len=CONFIG["sequence_length"])
                out.append((seq, label))
                counts[cls] += 1
            except Exception:
                continue

    return out, counts


def extract_pose_kps(frame_bgr: np.ndarray, pose_model):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = pose_model.process(frame_rgb)
    if res.pose_landmarks is None:
        return None
    return np.asarray([[lm.x, lm.y, lm.z] for lm in res.pose_landmarks.landmark], dtype=np.float32)


def load_source_rwf(root: Path):
    out = []
    counts = {"Fight": 0, "NonFight": 0}
    mapping = {"Fight": 1, "NonFight": 0}

    mp_pose = mp.solutions.pose
    with mp_pose.Pose(static_image_mode=False, model_complexity=1) as pose_model:
        for cls, label in mapping.items():
            cls_dir = root / "train" / cls
            if not cls_dir.exists():
                continue

            for vf in tqdm(sorted(cls_dir.rglob("*.avi")), desc=f"RWF {cls}", leave=False):
                cap = cv2.VideoCapture(str(vf))
                if not cap.isOpened():
                    continue
                fps = cap.get(cv2.CAP_PROP_FPS)
                fps = 25.0 if fps is None or fps <= 0 else fps
                step = max(1, int(round(fps / CONFIG["rwf_fps"])))

                feats = []
                prev = None
                frame_idx = 0
                while True:
                    ok, frame = cap.read()
                    if not ok:
                        break
                    if frame_idx % step == 0:
                        kps = extract_pose_kps(frame, pose_model)
                        if kps is None:
                            feats.append(zero_feature())
                            prev = None
                        else:
                            feats.append(build_frame_feature(kps, prev, 1.0))
                            prev = kps
                    frame_idx += 1
                cap.release()

                for seq in sliding_windows(feats, seq_len=CONFIG["sequence_length"], stride=CONFIG["stride"]):
                    out.append((seq, label))
                    counts[cls] += 1

    return out, counts


def group_ucf_frames(class_dir: Path):
    groups = defaultdict(list)
    pat = re.compile(r"^(.*)_([0-9]+)$")
    for fp in sorted(class_dir.rglob("*.png")):
        stem = fp.stem
        m = pat.match(stem)
        if m:
            groups[m.group(1)].append((int(m.group(2)), fp))
        else:
            groups[stem].append((0, fp))

    ordered = {}
    for name, pairs in groups.items():
        ordered[name] = [p for _, p in sorted(pairs, key=lambda x: x[0])]
    return ordered


def load_source_ucf(root: Path):
    out = []
    counts = {"Fighting": 0, "Assault": 0, "Normal Videos": 0, "Vandalism": 0}
    mapping = {"Fighting": 1, "Assault": 1, "Normal Videos": 0, "Vandalism": 1}

    mp_pose = mp.solutions.pose
    with mp_pose.Pose(static_image_mode=True, model_complexity=1) as pose_model:
        for cls, label in mapping.items():
            cls_dir = root / cls
            if not cls_dir.exists():
                continue

            grouped = group_ucf_frames(cls_dir)
            for _, frame_paths in tqdm(grouped.items(), desc=f"UCF {cls}", leave=False):
                feats = []
                prev = None
                for fp in frame_paths:
                    img = cv2.imread(str(fp))
                    if img is None:
                        continue
                    kps = extract_pose_kps(img, pose_model)
                    if kps is None:
                        feats.append(zero_feature())
                        prev = None
                    else:
                        feats.append(build_frame_feature(kps, prev, 1.0))
                        prev = kps
                if len(feats) == 0:
                    continue
                seq = to_fixed_sequence(feats, seq_len=CONFIG["sequence_length"])
                out.append((seq, label))
                counts[cls] += 1

    return out, counts

In [ ]:
# SECTION 7 - HybridCrowdDataset Class (Combine Sources + Augmentation)

class HybridCrowdDataset(torch.utils.data.Dataset):
    def __init__(self, sequences, labels, train_mode=False):
        self.sequences = [np.asarray(s, dtype=np.float32) for s in sequences]
        self.labels = [int(y) for y in labels]
        self.train_mode = train_mode

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        x = self.sequences[idx].copy()
        y = self.labels[idx]

        # Apply augmentation only for training split.
        if self.train_mode:
            # 50%: drop 3 random frames by replacing them with zeros.
            if random.random() < 0.5:
                drop_idx = random.sample(range(x.shape[0]), k=min(3, x.shape[0]))
                x[drop_idx, :] = 0.0

            # 30%: add Gaussian noise to keypoint dimensions.
            if random.random() < 0.3:
                noise = np.random.normal(0.0, 0.02, size=x[:, :99].shape).astype(np.float32)
                x[:, :99] += noise

            # 20%: horizontally flip x-coordinates only (every 3rd value in keypoint block).
            if random.random() < 0.2:
                x_idx = np.arange(0, 99, 3)
                x[:, x_idx] *= -1.0

        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

In [ ]:
# SECTION 8 - Dataset Statistics, Class Weights, and Stratified Splits

for key in ["godseye", "rwf2000", "ucf_crime"]:
    if not PATHS[key].exists():
        raise FileNotFoundError(f"Missing required dataset directory: {PATHS[key]}")

source1_samples, source1_counts = load_source_godseye(PATHS["godseye"])
source2_samples, source2_counts = load_source_rwf(PATHS["rwf2000"])
source3_samples, source3_counts = load_source_ucf(PATHS["ucf_crime"])

all_samples = source1_samples + source2_samples + source3_samples
if len(all_samples) == 0:
    raise RuntimeError("No sequences were built. Check dataset folders and formats.")

all_sequences = [s for s, y in all_samples]
all_labels = np.asarray([y for s, y in all_samples], dtype=np.int64)

print("Total sequences per source:")
print(f"  Source 1 GodsEye: {len(source1_samples)} | details={source1_counts}")
print(f"  Source 2 RWF-2000: {len(source2_samples)} | details={source2_counts}")
print(f"  Source 3 UCF-Crime: {len(source3_samples)} | details={source3_counts}")
print(f"Total sequences: {len(all_sequences)}")
print(f"Class balance (0=normal,1=anomaly): {dict(Counter(all_labels.tolist()))}")

# Compute class weights using required formula n_samples / (n_classes * class_counts).
class_counts = np.bincount(all_labels, minlength=2)
n_samples = len(all_labels)
n_classes = 2
class_weights = n_samples / (n_classes * np.maximum(class_counts, 1))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=CONFIG["device"])

print(f"Class counts: {class_counts.tolist()}")
print(f"Class weights: {class_weights.tolist()}")

# 80/10/10 split via two-stage stratified split with fixed random seed.
all_idx = np.arange(len(all_sequences))
train_val_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.10,
    stratify=all_labels,
    random_state=42,
)
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=(1.0 / 9.0),
    stratify=all_labels[train_val_idx],
    random_state=42,
)

In [ ]:
# SECTION 9 - DataLoader Construction for Train/Validation/Test

def pick_subset(indices):
    xs = [all_sequences[i] for i in indices]
    ys = [int(all_labels[i]) for i in indices]
    return xs, ys

x_train, y_train = pick_subset(train_idx)
x_val, y_val = pick_subset(val_idx)
x_test, y_test = pick_subset(test_idx)

train_dataset = HybridCrowdDataset(x_train, y_train, train_mode=True)
val_dataset = HybridCrowdDataset(x_val, y_val, train_mode=False)
test_dataset = HybridCrowdDataset(x_test, y_test, train_mode=False)

pin_memory = True if CONFIG["device"] == "cuda" else False
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=pin_memory)

xb, yb = next(iter(train_loader))
print(f"Train/Val/Test sizes: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")
print(f"Batch shape: {tuple(xb.shape)}, labels shape: {tuple(yb.shape)}")
assert xb.shape[1] == CONFIG["sequence_length"] and xb.shape[2] == 118

In [ ]:
# SECTION 10 - Manual LSTMCell Implementation (No nn.LSTM)

class LSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # Forget gate f_t controls retention of previous cell memory.
        self.forget_gate = nn.Linear(input_size + hidden_size, hidden_size)
        # Input gate i_t controls how much candidate info is written to memory.
        self.input_gate = nn.Linear(input_size + hidden_size, hidden_size)
        # Candidate gate g_t proposes new content for memory update.
        self.cell_gate = nn.Linear(input_size + hidden_size, hidden_size)
        # Output gate o_t controls how much memory is exposed as hidden state.
        self.output_gate = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat([x, h_prev], dim=1)
        f_t = torch.sigmoid(self.forget_gate(combined))
        i_t = torch.sigmoid(self.input_gate(combined))
        g_t = torch.tanh(self.cell_gate(combined))
        o_t = torch.sigmoid(self.output_gate(combined))

        c_next = f_t * c_prev + i_t * g_t
        h_next = o_t * torch.tanh(c_next)
        return h_next, c_next

In [ ]:
# SECTION 11 - CustomLSTM Stack (2 Layers + LayerNorm + Dropout)

class CustomLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = 256
        self.hidden2 = 128

        self.layer1 = LSTMCell(input_size=118, hidden_size=self.hidden1)
        self.layer2 = LSTMCell(input_size=self.hidden1, hidden_size=self.hidden2)

        self.norm1 = nn.LayerNorm(self.hidden1)
        self.norm2 = nn.LayerNorm(self.hidden2)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        device = x.device

        h1 = torch.zeros(batch_size, self.hidden1, device=device)
        c1 = torch.zeros(batch_size, self.hidden1, device=device)
        h2 = torch.zeros(batch_size, self.hidden2, device=device)
        c2 = torch.zeros(batch_size, self.hidden2, device=device)

        for t in range(seq_len):
            x_t = x[:, t, :]
            h1, c1 = self.layer1(x_t, h1, c1)
            h1 = self.norm1(h1)
            h1 = self.dropout(h1)

            h2, c2 = self.layer2(h1, h2, c2)
            h2 = self.norm2(h2)

        return h2

In [ ]:
# SECTION 12 - CrowdAnomalyClassifier Head + Confidence Method

class CrowdAnomalyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.temporal = CustomLSTM()
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        z = self.temporal(x)
        z = self.relu(self.fc1(z))
        z = self.dropout(z)
        z = self.relu(self.fc2(z))
        logits = self.fc3(z)
        return logits

    def get_confidence_score(self, logits_or_inputs):
        # Accept either logits tensor [B,2] or raw sequence tensor [B,T,118].
        if logits_or_inputs.dim() == 3:
            logits = self.forward(logits_or_inputs)
        else:
            logits = logits_or_inputs
        probs = torch.softmax(logits, dim=1)
        return probs[:, 1]

In [ ]:
# SECTION 13 - Training Utilities (Metrics, Early Stopping, Checkpointing, Curves)

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


def save_checkpoint(path, model, optimizer, epoch, val_f1, config):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_f1": val_f1,
            "CONFIG": config,
        },
        path,
    )


history = {
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": [],
}

best_checkpoint_path = PATHS["models"] / "best_checkpoint.pt"
best_weights_path = PATHS["models"] / CONFIG["model_name"]

In [ ]:
# SECTION 14 - Full Training Loop (Adam, CrossEntropy with Weights, CosineAnnealingLR, Grad Clipping)

model = CrowdAnomalyClassifier().to(CONFIG["device"])
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

best_val_loss = float("inf")
no_improve_epochs = 0

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_losses = []

    for xb, yb in train_loader:
        xb = xb.to(CONFIG["device"])
        yb = yb.to(CONFIG["device"])

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        train_losses.append(float(loss.item()))

    model.eval()
    val_losses = []
    val_true = []
    val_pred = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(CONFIG["device"])
            yb = yb.to(CONFIG["device"])

            logits = model(xb)
            vloss = criterion(logits, yb)
            preds = torch.argmax(logits, dim=1)

            val_losses.append(float(vloss.item()))
            val_true.extend(yb.cpu().numpy().tolist())
            val_pred.extend(preds.cpu().numpy().tolist())

    train_loss = float(np.mean(train_losses)) if train_losses else 0.0
    val_loss = float(np.mean(val_losses)) if val_losses else 0.0
    m = compute_metrics(val_true, val_pred) if len(val_true) > 0 else {"accuracy":0.0,"precision":0.0,"recall":0.0,"f1":0.0}

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(m["accuracy"])
    history["val_f1"].append(m["f1"])

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"epoch={epoch:03d} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={m['accuracy']:.4f} val_f1={m['f1']:.4f} lr={lr_now:.6f}")

    if val_loss < (best_val_loss - CONFIG["min_delta"]):
        best_val_loss = val_loss
        no_improve_epochs = 0
        save_checkpoint(best_checkpoint_path, model, optimizer, epoch, m["f1"], CONFIG)
        torch.save(model.state_dict(), best_weights_path)
    else:
        no_improve_epochs += 1

    if no_improve_epochs >= CONFIG["patience"]:
        print(f"Early stopping triggered at epoch {epoch}.")
        break